In [1]:
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
embeddings = np.load(
    "../data/processed/paper_embeddings.npy"
)

papers = pd.read_parquet(
    "../data/processed/embedded_papers.parquet"
)

print("Embeddings:", embeddings.shape)
print("Papers:", papers.shape)

Embeddings: (767, 384)
Papers: (767, 19)


In [3]:
similarity_matrix = cosine_similarity(embeddings)

print("Similarity matrix:", similarity_matrix.shape)

Similarity matrix: (767, 767)


In [4]:
threshold = 0.65

G = nx.Graph()

for i in range(len(papers)):
    G.add_node(
        i,
        paper_id=papers.iloc[i]["paper_id"],
        title=papers.iloc[i]["title"],
        year=papers.iloc[i]["year"],
        citation_count=papers.iloc[i]["citation_count"]
    )

for i in range(len(papers)):
    for j in range(i + 1, len(papers)):
        similarity = similarity_matrix[i, j]

        if similarity >= threshold:
            G.add_edge(
                i,
                j,
                weight=float(similarity)
            )

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 767
Edges: 11064


In [5]:
degrees = [degree for _, degree in G.degree()]

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Average degree:", np.mean(degrees))
print("Maximum degree:", max(degrees))

Nodes: 767
Edges: 11064
Average degree: 28.85006518904824
Maximum degree: 193


In [6]:
# Basic graph statistics

print("Number of connected components:",
      nx.number_connected_components(G))

components = list(nx.connected_components(G))
component_sizes = sorted(
    [len(c) for c in components],
    reverse=True
)

print("Largest component:", component_sizes[0])
print("Smallest component:", component_sizes[-1])
print("Top 10 component sizes:", component_sizes[:10])

Number of connected components: 113
Largest component: 646
Smallest component: 1
Top 10 component sizes: [646, 4, 2, 2, 2, 2, 2, 2, 1, 1]


In [7]:
# Graph density

print("Graph density:", nx.density(G))

Graph density: 0.037663270481786214


In [8]:
import pickle

with open("../data/processed/research_graph.pkl", "wb") as f:
    pickle.dump(G, f)

print("Graph saved successfully.")

Graph saved successfully.
